# 020 — Disaggregation setup & analyses

Sets up and runs the **IML-based** seismic hazard disaggregations for the WP1 sites.

1. Loads the target intensity levels (IMLs) chosen in `017-disagg_imls_for_msa_stripes.ipynb`.
2. Looks up the **classical parent**: nb 003's PSHA `AvgSA_03_psha_eps4`, the base hazard
   calculation, which stores the rupture data. This notebook runs no PSHA.
3. Runs (or reuses) **one disaggregation child** on that parent, which disaggregates every
   site at every target IML $x$ *and* at $x + \delta x$.
4. Records every `calc_id` in `wp1/disagg_manifest.json` so the datastores can be
   obtained programmatically later.

The output is the same as before — the disaggregation of all IMLs at all 60 sites — with
no classical run at all (nb 003's is reused) instead of one per IML. All the engine-facing code is in
`phd_project/scripts/oq_multi_iml_disagg.py`, shared with nb 021, nb 072 and nb 065.

## Prerequisites

`003-psha_setup_and_analyses.ipynb` **must have been run first**. This notebook copies
nothing — it reuses what already sits in `hazard_models/eshm20/wp1/`:

| File | Provenance |
|---|---|
| `source_model_logic_tree_eshm20.xml` | untouched ESHM20, placed manually |
| `source_models/` | untouched ESHM20, placed manually |
| `gmpe_logic_tree_AvgSA_0to{3,6}_median_branch.xml` | simplified by hand (single median branch per TRT) |
| `site_model_casestudy_sites.csv` | written from `/results` by notebook 003 |

The target IMLs come from `data_processed/03_site_hazard/AvgSA_03_imls_for_disaggregation.csv`.

## How it works: one parent, cheap children

The expensive part of a disaggregation is finding which ruptures can reach each site and
how strongly (distances, GMM mean $\mu$ and $\sigma$). None of that depends on the
intensity level. The level only enters at the very end:

$$
P(IM > x \mid \text{rup}) = 1 - \Phi\!\left(\frac{\ln x - \mu_{\text{rup}}}{\sigma_{\text{rup}}}\right)
$$

So the **parent** (nb 003's PSHA, whose config carries `max_sites_disagg = 60`) has
already run and stored those rupture "contexts". The **child** (`hazard_calculation_id = parent`) reads them
back and evaluates as many levels per site as we like. An analogy: the parent works out,
once, which earthquakes can reach each site and how hard; the child just reads off, for
any shaking level we name, which of those earthquakes are responsible.

The engine already loops over a per-site level array `hmap3` of shape (sites, IMTs,
levels). The child ini sets `poes_disagg` only as a **placeholder** that fixes the number
of levels; the project subclass `MultiIMLDisaggCalculator` replaces `hmap3` with our levels
in the child's own new datastore, before any task reads it. The child keeps the parent's
25-level IMTL grid, which is what lets it chain at all: the engine ranks realisations by
comparing the parent curve with the child's grid, and a 1-level `iml_disagg` child broke
that ranking — the reason the old jobs were standalone.

## Why two levels per IML: occurrence

OpenQuake disaggregates **exceedance**, $P(m \mid IM \ge x)$. Record selection needs
**occurrence**, $P(m \mid IM = x)$. Bradley (2010, *EESD* 39:1321–1342, Eq. 7, p. 1326)
obtains it from two exceedance disaggregations, at $x$ and at $x + \delta x$:

$$
P(m \mid X = x) \approx \frac{\nu_m(x) - \nu_m(x+\delta x)}{\nu(x) - \nu(x+\delta x)},
\qquad \nu(x) = \sum_m \nu_m(x), \qquad \delta x = 0.001\,x
$$

where $\nu_m(x)$ is the annual rate at which bin $m$ (TRT × M × R × ε) produces
$IM \ge x$. The old post-processing used the exceedance share at $x$ in *both* terms, which
cancels to the exceedance share: every `P(m|X=x)` written before this change was
$P(m \mid X > x)$. With 38 target IMLs the child therefore carries **76 levels per site**.

## The old per-IML jobs

The 38 standalone `iml_disagg` jobs (`AvgSA_03_disagg_eps4_<iml>`) and their configs are
**kept**, untouched, as the regression reference: §4 checks that the child reproduces each
of them. `oq_runner.write_disagg_config` (the stock route) is kept as the fallback.

## Adding IMLs later

Run a **new child** on the same parent with only the new levels (e.g. name it
`AvgSA_03_disagg_eps4_imlgrid_c01`). nb 021 merges every `..._imlgrid*` child. No classical
re-run is needed unless the source model, GMM logic tree, site model, IMT or truncation
level changes — but the parent datastore must stay in this machine's `oqdata`. If you change
`MAX_LEVELS_PER_CHILD`, remove the superseded child entries from the manifest: nb 021
refuses to merge two children that supply the same IML.

## Runtime

The memory needed by the engine's accumulator grows with the number of levels in a child
(~76× one old job's `The AccumDict will require X GB` log line). OpenQuake refuses to start
if it cannot fit it. If that happens, set `MAX_LEVELS_PER_CHILD` to split the levels over
several children (each child is still far cheaper than a classical run).

`num_rlzs_disagg = 0` (all 21 SSC realizations) is **required**: the mean disaggregation is
only computed when more than one realization is kept.

Cells starting with `############### Check Cell ############` are one-off checks: once they
have run on the analysis machine and pass, delete them.

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0. Setup & parameters

In [22]:
import numpy as np
import pandas as pd

from phd_project.config import config
from phd_project.scripts import oq_runner
from phd_project.scripts import oq_multi_iml_disagg as mid

cfg = config.load_config()

In [23]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
WP1_DIR = cfg["hazard_models"]["eshm20_wp1"]
MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_disagg_manifest"]
SITE_MODEL_FP = cfg["hazard_models"]["eshm20_wp1_site_model"]

# The IM definition and truncation level carried forward (AvgSA[0,6] and eps 3/5 are
# not considered beyond this point).
IM = "03"
EPS = 4
IML_FILE = cfg["proc_data"]["disagg_imls_AvgSA_03"]

# 0 = keep all realizations. Required, not optional: the mean disaggregation
# (disagg-stats) is only computed when more than one realization is kept.
NUM_RLZS_DISAGG = 0

# Occurrence pair: every IML x is also disaggregated at x * (1 + DELTA)
# (Bradley 2010, Eq. 7). 0.001 as in the old code, so only the formula changes.
DELTA = mid.DELTA_DEFAULT

# The parent is nb 003's PSHA for this IM and truncation level ("AvgSA_03_psha_eps4"),
# recorded in nb 003's manifest: it is the base hazard calculation, so this notebook
# runs no PSHA of its own.
PSHA_MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_psha_manifest"]
PARENT_NAME = mid.parent_name(IM, EPS)
# The child keeps the AvgSA_03_disagg_eps4_ prefix, so
# oqhelpers.assert_shared_provenance (nb 021) checks it with the other disagg runs.
CHILD_NAME = f"AvgSA_{IM}_disagg_eps{EPS}_imlgrid"

# None = all 2 x 38 = 76 levels in one child. An integer splits them over several
# children of at most that many levels (named CHILD_NAME_c00, _c01, ...), if one child
# needs more memory than the machine has.
MAX_LEVELS_PER_CHILD = None

# DRY_RUN:     write configs and report what would run, but launch nothing.
# FORCE_RERUN: re-run even when the inputs are unchanged.
# NEW_WINDOW:  give each calculation its own console window so its progress can be
#              watched (Windows only; output is teed to WP1_DIR/logs/<name>.log
#              either way). Runs stay sequential regardless.
DRY_RUN = False
FORCE_RERUN = False
NEW_WINDOW = True
# -----------------------------------------------------------------------------

N_SITES = len(pd.read_csv(SITE_MODEL_FP))
print(f"wp1 dir:  {WP1_DIR}")
print(f"manifest: {MANIFEST_FP}")
print(f"{N_SITES} sites; parent {PARENT_NAME}; child {CHILD_NAME}")
print(f"DRY_RUN={DRY_RUN}, FORCE_RERUN={FORCE_RERUN}, NEW_WINDOW={NEW_WINDOW}")

wp1 dir:  C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1
manifest: C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1\disagg_manifest.json
60 sites; parent AvgSA_03_psha_eps4; child AvgSA_03_disagg_eps4_imlgrid
DRY_RUN=False, FORCE_RERUN=False, NEW_WINDOW=True


## 1. Load the target IMLs

Written by `017-disagg_imls_for_msa_stripes.ipynb`. Each IML is used at 6 significant
figures: that is the key nb 021 files it under, and the exact value the old per-IML jobs
disaggregated at (their ini held `f"{iml:.6g}"`), so the regression check in §4 compares
the same level.

In [24]:
if not IML_FILE.is_file():
    raise FileNotFoundError(f"no IML file at {IML_FILE} - run 017 first")
imls = oq_runner.load_imls(IML_FILE)
keys = [float(oq_runner._iml_str(v)) for v in imls]       # 6 s.f.: key AND level
assert len(set(keys)) == len(keys), "two target IMLs coincide at 6 s.f."
print(f"AvgSA {IM}: {len(keys)} imls from {IML_FILE.name}")
print(f"           {keys}")

AvgSA 03: 38 imls from AvgSA_03_imls_for_disaggregation.csv
           [0.1645, 0.2109, 0.2423, 0.27, 0.285, 0.31, 0.33, 0.3434, 0.36, 0.3649, 0.3882, 0.4, 0.4028, 0.4221, 0.4393, 0.45, 0.4635, 0.4898, 0.5, 0.5127, 0.5268, 0.55, 0.5971, 0.6049, 0.65, 0.6947, 0.7139, 0.7473, 0.7892, 0.8, 0.8904, 0.9122, 0.95, 1.0112, 1.0475, 1.15, 1.3368, 1.8039]


## 2. The classical parent

The parent is nb 003's PSHA `AvgSA_03_psha_eps4`, read from `psha_manifest.json`. Nothing
is run here: `require_parent` only checks that the datastore is a 60-site classical run
with `max_sites_disagg = 60` and stored rupture contexts. If it fails, re-run nb 003.

In [25]:
# nb 003's PSHA, checked fit to be a parent (classical, 60 sites, max_sites_disagg = 60,
# rupture contexts stored). Raises with "re-run nb 003" if not.
parent_calc_id = mid.require_parent(PSHA_MANIFEST_FP, PARENT_NAME, N_SITES)
print(f"parent {PARENT_NAME}: calc_id = {parent_calc_id}")

parent AvgSA_03_psha_eps4: calc_id = 143


## 3. The child: every IML at $x$ and $x + \delta x$

38 targets × {$x$, $x+\delta x$} = 76 levels at every site ($N=60$, $M=1$, $P=76$). The
levels table (`wp1/levels_<name>.csv`, plus the `.npy` the calculator loads) is written
next to the ini; its hash is part of the child's fingerprint, so changing an IML, `DELTA`
or the parent makes the child re-run.

In [28]:
targets = pd.DataFrame([(s, k, k) for s in range(N_SITES) for k in keys],
                       columns=["site", "key", "x"])
chunks = mid.split_targets(targets, MAX_LEVELS_PER_CHILD)

child_calc_ids, child_tables = {}, {}
if parent_calc_id is None:
    print("parent not run yet (DRY_RUN?) - the child ini needs its calc_id; nothing written")
else:
    for i, chunk in enumerate(chunks):
        name = CHILD_NAME if len(chunks) == 1 else f"{CHILD_NAME}_c{i:02d}"
        table = mid.build_levels_table(chunk, DELTA)
        ini = mid.write_child_config(
            WP1_DIR / f"config_{name}.ini", parent_calc_id=parent_calc_id,
            parent_name=PARENT_NAME, n_levels=int(table.p.max()) + 1,
            description=mid.child_description(name),
            levels_file=f"{mid.levels_file_stem(name)}.npy",
            im=IM, truncation_level=EPS, num_rlzs_disagg=NUM_RLZS_DISAGG)
        child_tables[name] = table
        child_calc_ids[name] = mid.run_or_reuse_child(
            name, ini, table, N_SITES, WP1_DIR, MANIFEST_FP, parent_name=PARENT_NAME,
            parent_manifest_fp=PSHA_MANIFEST_FP,
            force_rerun=FORCE_RERUN, dry_run=DRY_RUN, new_window=NEW_WINDOW)
        print(f"  {name}: {table.p.max() + 1} levels per site -> calc_id {child_calc_ids[name]}")
child_calc_ids

[oq] 'AvgSA_03_disagg_eps4_imlgrid' inputs changed since calc_id=141:
  - config: file config_AvgSA_03_disagg_eps4_imlgrid.ini (2993 bytes) [fa26e576f50cdb50f68c5dda79e14588280261b92ce9e33fc2d21f36e8232472] -> file config_AvgSA_03_disagg_eps4_imlgrid.ini (2984 bytes) [6630a95d04de192221fc48387c16f4ecef2122ffa1e5f899b126d01eb0cf8beb]
  - parent_calc_id: int=138 [d6a4031733610bb080d0bfa794fcc9dbdcff74834aeaab7c6b927e21e9754037] -> int=143 [d6f0c71ef0c88e45e4b3a2118fcb83b0def392d759c901e9d755d0e879028727]
  - parent_inputs: str '{"config": {"hash": "28f6c54ef57d26e162bf6281156ceca2bc677dc9bb75118f7536920e0da2e481", "summary": "file config_AvgSA_03_disagg_eps4_parent.ini (1785 bytes)"}, "gmpe_lt": {"hash": "469f0c8fb6ddd0862c83f3ba283b9119018432bf24927f74e066e270e3155169", "summary": "file gmpe_logic_tree_AvgSA_0to3_median_branch.xml (4452 bytes)"}, "pickagm_version": {"hash": "0.1.0", "summary": "pickagm version"}, "site_model": {"hash": "e388b6bb36bb968be8389fc861c1e8f3d6e3fd1e510c405aac

{'AvgSA_03_disagg_eps4_imlgrid': 150}

In [33]:
table

,site,imt,p,key,role,iml_g,key_kind
0,0,AvgSA,0,0.1645,x,0.164500,float
1,0,AvgSA,1,0.1645,x+dx,0.164664,float
2,0,AvgSA,2,0.2109,x,0.210900,float
3,0,AvgSA,3,0.2109,x+dx,0.211111,float
4,0,AvgSA,4,0.2423,x,0.242300,float
...,...,...,...,...,...,...,...
4555,59,AvgSA,71,1.1500,x+dx,1.151150,float
4556,59,AvgSA,72,1.3368,x,1.336800,float
4557,59,AvgSA,73,1.3368,x+dx,1.338137,float
4558,59,AvgSA,74,1.8039,x,1.803900,float


## 4. Manifest

`wp1/disagg_manifest.json` maps each analysis to its `calc_id`, together with the content
hashes of the inputs that produced it. It is small text and **git-tracked**. A child entry
also records its parent, its levels file and the engine version and fork commit it ran
with.

Downstream notebooks should read it with `oq_runner.load_calc_ids(MANIFEST_FP)` (or
`mid.child_entries(...)` for the children) rather than hardcoding integers.

In [32]:
manifest = oq_runner.load_manifest(MANIFEST_FP)

if not manifest:
    print(f"no manifest yet at {MANIFEST_FP} (nothing has been run)")
else:
    rows = [{"analysis": name,
             "calc_id": e["calc_id"],
             "kind": e.get("kind", "stock"),
             "parent_calc_id": e.get("parent_calc_id"),
             "n_levels": e.get("n_levels", 1),
             "config": e["config"],
             "engine": e.get("_meta", {}).get("engine_version", ""),
             "written_at": e["_meta"]["written_at"]}
            for name, e in manifest.items()]
    display(pd.DataFrame(rows).sort_values(["kind", "analysis"]).reset_index(drop=True))

,analysis,calc_id,kind,parent_calc_id,n_levels,config,engine,written_at
0,AvgSA_03_disagg_eps4_imlgrid,150,multi_iml_child,143.0,76,config_AvgSA_03_disagg_eps4_imlgrid.ini,3.27.0,2026-10-29T07:00:09.157557+00:00
1,AvgSA_03_disagg_eps4_smoke,149,multi_iml_child,143.0,4,config_AvgSA_03_disagg_eps4_smoke.ini,3.27.0,2026-10-29T06:54:44.762212+00:00
2,AvgSA_03_disagg_eps4_smoke_perturb,151,multi_iml_child,143.0,4,config_AvgSA_03_disagg_eps4_smoke_perturb.ini,3.27.0,2026-10-29T07:06:27.052614+00:00
3,AvgSA_03_disagg_eps4_0pt165,102,stock,NaN,1,config_AvgSA_03_disagg_eps4_0pt165.ini,,2026-08-25T08:42:33.146546+00:00
4,AvgSA_03_disagg_eps4_0pt211,103,stock,NaN,1,config_AvgSA_03_disagg_eps4_0pt211.ini,,2026-08-25T08:52:42.496329+00:00
5,AvgSA_03_disagg_eps4_0pt242,104,stock,NaN,1,config_AvgSA_03_disagg_eps4_0pt242.ini,,2026-08-25T09:00:49.704951+00:00
6,AvgSA_03_disagg_eps4_0pt270,89,stock,NaN,1,config_AvgSA_03_disagg_eps4_0pt270.ini,,2026-07-29T08:37:31.513423+00:00
7,AvgSA_03_disagg_eps4_0pt285,90,stock,NaN,1,config_AvgSA_03_disagg_eps4_0pt285.ini,,2026-07-29T08:48:21.814373+00:00
8,AvgSA_03_disagg_eps4_0pt310,91,stock,NaN,1,config_AvgSA_03_disagg_eps4_0pt310.ini,,2026-07-29T08:58:11.991078+00:00
9,AvgSA_03_disagg_eps4_0pt330,92,stock,NaN,1,config_AvgSA_03_disagg_eps4_0pt330.ini,,2026-07-29T09:07:17.119958+00:00
